In [5]:
import re, pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /home/dasha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
df = pd.read_excel("narratives_all_3009.xlsx")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123273 entries, 0 to 123272
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   id              123273 non-null  object        
 1   message_id      123273 non-null  int64         
 2   date            123273 non-null  datetime64[ns]
 3   message         123273 non-null  object        
 4   id_channel      123273 non-null  int64         
 5   message_vector  123263 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 5.6+ MB


In [8]:

def clean(txt: str) -> str:
    txt = re.sub(r'http\S+|www\.\S+', ' ', txt)        # ссылки
    txt = re.sub(r'[@#]\w+', ' ', txt)                 # @, #
    txt = re.sub(r'\s+', ' ', txt).strip()
    return txt

df['msg_clean'] = df['message'].astype(str).map(clean)

# отсечь "пустые" и очень короткие
df = df[df['msg_clean'].str.split().str.len().fillna(0) >= 7]

# # примитивное удаление дубликатов/почти-дубликатов (TF-IDF + косинус)
# sub = df.sample(min(20000, len(df)), random_state=42)
# vec = TfidfVectorizer(min_df=3, max_df=0.6).fit(sub['msg_clean'])
# m = vec.transform(df['msg_clean'])
# # считаем схожесть соседей в пределах канала и суток (ускоряем через группировку при необходимости)
# # простая версия: точные дубли
# df = df.drop_duplicates(subset=['msg_clean'])

In [9]:
# ключи "экономики" (стемы/основы, чтобы покрыть склонения)
ECON_KEEP = r"""
(эконом|инфляц|безработиц|ВВП|ВПП|бюджет|дефицит|профицит|
налог|НДС|акциз|пошлин|тариф|квот|субсид|
ЦБ|центробанк|ключев\w* ставк|ставк[аи]|процентн\w* ставк|
рубл|доллар|евро|курс|валют|
экспорт|импорт|внешне|торгов|таможн|растамож|
санкци|эмбарг|ограничен|CFSP|
нефть|газ|уголь|энергетик|ОПЕК|
индекс|бирж|Мосбирж|NASDAQ|S&P|облигац|ОФЗ|доходност|
инвестиц|капитальн\w* вложен|IPO|IPO|M&A|
кредит|ипотек|займ|банковск|
зарплат|пенси|реальны\w* доход|
промышленн|производств|ПМИ|розниц|опт|
логистик|поставк|порт|жд|контейнер|
таможенн\w* союз|ЕАЭС|пошлин|квот|сертификат происхожд|
сельхоз|агро|зерн|экспортн\w* пошлин|минсельхоз
)
"""

# явные шумы (реклама/мемы/развлечения). Крипту можно вынести в отдельную метку.
NOISE_DROP = r"""
(розыгрыш|конкурс|подписывай|подписк|ставк[аи]\s+на\s+спорт|букмек|казин|промокод|
гороскоп|погода|мем|шутк|анекдот|юмор|рецепт|кино|сериал|шоу|
ваканси|резюме|ищем\s+сотрудник|мерч|
продаж[аи]\s+курсов|вебинар\s+по|реклам|спонсорск|
18\+|NSFW)
"""

# сообщения, которые оставляем, если есть хотя бы 1 эконом-ключ и нет стоп-слов
keep_re = re.compile(ECON_KEEP, re.IGNORECASE | re.VERBOSE)
drop_re = re.compile(NOISE_DROP, re.IGNORECASE | re.VERBOSE)

mask_rule = df['msg_clean'].str.contains(keep_re) & ~df['msg_clean'].str.contains(drop_re)
df_rule = df[mask_rule].copy()


In [10]:
df_rule.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68706 entries, 0 to 123270
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              68706 non-null  object        
 1   message_id      68706 non-null  int64         
 2   date            68706 non-null  datetime64[ns]
 3   message         68706 non-null  object        
 4   id_channel      68706 non-null  int64         
 5   message_vector  68706 non-null  object        
 6   msg_clean       68706 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 4.2+ MB


In [13]:
import re

# 1) жёсткие стопы
HARD_DROP = {
    # силовой/уголовка/ЧП/война
    "правоохран","уголовн","следств","задержан","арест","приговор","обыск","мошеннич",
    "дтп","пожар","взрыв","катастроф","мобилизац","обстрел","фортификацион","пригранич",
    # внеэконом разное
    "скандал","конфликт","митинг","праздник","юбиле","конкурс","розыгрыш","мем",
    "промокод","подписк","реклама","спонсор"
}

# 2) экономические триггеры (должен быть хотя бы один)
ECON_TRIGGERS = {
    # макро
    "ввп","инфляц","безработиц","производств","промышленн","пми","розниц","опт","индекс",
    # денежно-кредитная / финрынки
    "цб","центробанк","ключев","ставк","процент","облигац","офз","курс","девальв","ревальв",
    # фискалка / торговая политика
    "бюджет","налог","ндс","акциз","субсид","тариф","пошлин","квот","лицензир","санкци","эмбарг",
    # внешняя торговля и логистика
    "экспорт","импорт","внешне","таможн","сертификат","логистик","жд","порт","контейнер","транзит","еаэс",
    # сырьё/энергетика
    "нефть","газ","уголь","энергетик","опек","brent","urals","добыч",
    # корп-финансы
    "выручк","прибыл","убыток","ebit","дивиденд","ipo","m&a","объединен","поглощен","инвестиц","капвложен"
}

# 3) маркеры события/динамики (требуется один из: ACTION or QUANT)
ACTION = {
    "повыс","сниз","подня","сохрани","измен","установ","ввел","отмен","ужесточ","смягч","подписал",
    "одобр","запрет","разреш","продлил","ввез","вывез","вырос","сократ","упал","увелич","сокращен"
}

# 4) количественные маркеры: проценты, валюта, объёмы
RE_QUANT = re.compile(r"(%|\b[0-9]+([.,][0-9]+)?\s*(млн|млрд|тыс|руб|₽|\$|usd|евро|eur|тонн|баррел|bbl|квт|мвт)\b)")

def is_econ_candidate(stems: set[str], raw_text: str) -> bool:
    # HARD DROP
    if stems & HARD_DROP:
        return False
    # MUST: эконом-триггер
    if not (stems & ECON_TRIGGERS):
        return False
    # MUST: действие/динамика или количественный маркер
    has_action = bool(stems & ACTION)
    has_quant  = bool(RE_QUANT.search(raw_text.lower()))
    if not (has_action or has_quant):
        return False
    # Доп. защита: если это «курс/рубль/доллар» без динамики — не пускаем
    only_fx = (stems & {"курс","рубл","доллар","евро"}) and not (has_action or has_quant)
    if only_fx:
        return False
    return True


In [15]:
def stems_from_vector(d):  # d = message_vector: dict{stem: id or weight}
    return set(map(str.lower, d.keys()))

mask = []
for msg, vec in zip(df_rule['message'], df_rule['message_vector']):
    stems = stems_from_vector(vec) if isinstance(vec, dict) else set()
    mask.append(is_econ_candidate(stems, msg))

df_econ = df_rule[mask].copy()


df_econ = df_rule[mask].copy()


In [19]:
df_rule.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68706 entries, 0 to 123270
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              68706 non-null  object        
 1   message_id      68706 non-null  int64         
 2   date            68706 non-null  datetime64[ns]
 3   message         68706 non-null  object        
 4   id_channel      68706 non-null  int64         
 5   message_vector  68706 non-null  object        
 6   msg_clean       68706 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 4.2+ MB


In [20]:
news = df_rule["message"].dropna().astype(str).tolist()

In [21]:
assert isinstance(news, list)
assert all(isinstance(a, str) for a in news)

In [22]:
from umap import UMAP

umap_model = UMAP(
    n_neighbors=20, n_components=8, min_dist=0.1, metric="cosine", random_state=42
)

In [23]:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(
    min_cluster_size=25, 
    min_samples=5, 
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)


In [24]:
import spacy
from sklearn.feature_extraction.text import CountVectorizer

nlp = spacy.load("ru_core_news_sm")

russian_stopwords = list(nlp.Defaults.stop_words)

vectorizer_model = CountVectorizer(
    stop_words=russian_stopwords, min_df=2, ngram_range=(1, 2)
)

In [25]:
model_name = "sberbank-ai/sbert_large_nlu_ru"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # last hidden state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size())
    return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1)


class HFEmbedder:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def embed_documents(self, documents, verbose=False):
        self.model.eval()
        all_embeddings = []
        batch_size = 32
        for i in tqdm(range(0, len(documents), batch_size), disable=not verbose):
            batch = documents[i : i + batch_size]
            encoded = self.tokenizer(
                batch, padding=True, truncation=True, return_tensors="pt"
            ).to(self.device)
            with torch.no_grad():
                model_output = self.model(**encoded)
            embeddings = mean_pooling(model_output, encoded["attention_mask"])
            all_embeddings.append(embeddings.cpu().numpy())
        return np.vstack(all_embeddings)

    def embed_words(self, words, verbose=False):
        return self.embed_documents(words, verbose=verbose)

In [ ]:
from openai import OpenAI as OpenAIClient
from bertopic.representation import (
    KeyBERTInspired,
    MaximalMarginalRelevance,
    PartOfSpeech,
    OpenAI,
)



# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
pos_model = PartOfSpeech("ru_core_news_md")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

client = OpenAIClient(
    api_key=" ", base_url=" "
)

openai_model = OpenAI(
    client=client,
    model="deepseek-chat",
    chat=True,
    prompt="""
Документы:
[DOCUMENTS]

Ключевые слова: [KEYWORDS]

Назови краткое, ёмкое название темы (до 5 слов) в формате:
topic: <название темы>
""",
)

representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
    "POS": pos_model,
    "DeepSeek": openai_model,
}

In [27]:
embedding_model = HFEmbedder(model, tokenizer, device)

topic_model = BERTopic(
    # Pipeline models
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    # Hyperparameters
    top_n_words=10,
    verbose=True,
    calculate_probabilities=True,
)

topics, probs = topic_model.fit_transform(news)

2025-09-30 12:56:37,784 - BERTopic - Embedding - Transforming documents to embeddings.


Batches: 100%|██████████| 2148/2148 [02:04<00:00, 17.28it/s]
2025-09-30 12:58:44,307 - BERTopic - Embedding - Completed ✓
2025-09-30 12:58:44,310 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-30 12:59:58,845 - BERTopic - Dimensionality - Completed ✓
2025-09-30 12:59:58,856 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-30 13:01:09,909 - BERTopic - Cluster - Completed ✓
2025-09-30 13:01:09,923 - BERTopic - Representation - Extracting topics from clusters using representation models.
100%|██████████| 174/174 [06:12<00:00,  2.14s/it]
2025-09-30 13:08:11,618 - BERTopic - Representation - Completed ✓


In [28]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,DeepSeek,Representative_Docs
0,-1,38793,-1_россии_сша_заявил_новости,"[россии, сша, заявил, новости, риа, риа новост...","[российской, российского, российских, россиян,...","[россии, сша, заявил, новости, риа, риа новост...","[новости, рублей, года, млрд, млн, области, сл...",[Российская экономика в условиях СВО],[Казахстан обошёл Россию по ключевому экономич...
1,0,4033,0_года_году_2024_2025,"[года, году, 2024, 2025, руб, млрд, год, рубле...","[2024 года, 2025 года, россиян, 2024 году, 202...","[года, году, 2024, 2025, руб, млрд, год, рубле...","[года, году, 2024, 2025, млрд, год, рублей, 20...",[Прогнозы экономики России],[👍 Что было сегодня\n\n🖨 Яндекс: МСФО 1к2025\n...
2,1,1773,1_аэс_оаэ_заявил_пмэф,"[аэс, оаэ, заявил, пмэф, электроэнергии, новос...","[минэкономразвития, электроэнергии, аэропорт, ...","[аэс, оаэ, заявил, пмэф, электроэнергии, новос...","[электроэнергии, новости, аэропорта, рейсов, с...",[Энергетика и транспорт России],"[Онлайн-регистрация на рейсы ""Аэрофлота"" досту..."
3,2,1652,2_цб_цен_цены_центр,"[цб, цен, цены, центр, россии, цифровой, центр...","[цифрового рубля, церкви, си цзиньпин, ценност...","[цб, цен, цены, центр, россии, цифровой, центр...","[цен, цены, центр, цифровой, центра, года, гла...",[Рост цен и энергокризис],[👍 Главное за неделю: макро и политика\n\n🏦 ЦБ...
4,3,1474,3_bankrollo_рублей_тысяч_тысяч рублей,"[bankrollo, рублей, тысяч, тысяч рублей, млн р...","[рублей bankrollo, bankrollo власти, bankrollo...","[bankrollo, рублей, тысяч, тысяч рублей, млн р...","[рублей, тысяч, млн, россиян, года, лет, млрд,...",[Финансовые реалии России],[В России цены на iPhone 17 будут доходить до ...
...,...,...,...,...,...,...,...,...,...
169,168,25,168_новости каналы_каналы_подписаться риа_подп...,"[новости каналы, каналы, подписаться риа, подп...","[риа новости, новости каналы, россии задержанн...","[новости каналы, каналы, подписаться риа, подп...","[каналы, суда, подозрению, наркотиков, киберпр...",[Аресты россиян и крушение Ан-24],[❗️Спасатели отправились в Тындинский район Ам...
170,169,25,169_русагро_ragr_мошковича_основателя русагро,"[русагро, ragr, мошковича, основателя русагро,...","[основателя русагро, основателю русагро, русаг...","[русагро, ragr, мошковича, основателя русагро,...","[основателя, мошенничестве, делу, злоупотребле...",[Арест основателя Русагро Мошковича],[#RAGR\nСУД ПРОДЛИЛ ДО КОНЦА НОЯБРЯ АРЕСТ ОСНО...
171,170,25,170_truth social_truth_social_трамп,"[truth social, truth, social, трамп, написал, ...","[соцсети truth, президентом россии, сети truth...","[truth social, truth, social, трамп, написал, ...","[президент, американский, социальной сети, соц...",[Трамп о России и Украине],[⚡️Американский президент Дональд Трамп подал ...
172,171,25,171_ozon_маркетплейс_маркетплейс ozon_27 декабря,"[ozon, маркетплейс, маркетплейс ozon, 27 декаб...","[маркетплейс ozon, ozon впервые, ozon мосбирже...","[ozon, маркетплейс, маркетплейс ozon, 27 декаб...","[маркетплейс, 27 декабря, товаров, товары, ком...",[Редомициляция Ozon в Россию],[#AFKS #OZON\nАкционеры Ozon 27 декабря рассмо...


In [ ]:
# topic_model.get_topic_info().to_excel('топики_отфильтрованных_новостей.xlsx', index=False)

In [32]:
import numpy as np
TARGET_TOPIC = 60
# проверить, что ID существует
assert 60 in topic_model.get_topics().keys(), "Топик 60 отсутствует в модели"

# опционально убедиться по имени/ключам
info = topic_model.get_topic_info()
print(info[info["Topic"]==60][["Topic","Count","Name"]])

# если хочешь находить по шаблону (на случай переобучения и смены ID):
# cand = info[info["Name"].str.contains(r"ключев(ая|ую)\s+ставк|bloomberg", case=False, regex=True)]
# TARGET_TOPIC = int(cand.iloc[0]["Topic"])


    Topic  Count                                        Name
61     60     95  60_bloomberg_ставку_ключевую ставку_ставки


In [35]:
doc_info = topic_model.get_document_info(news)
doc_info['row_id'] = df_rule.index.to_numpy()

mask_exact = (doc_info['Topic'] == TARGET_TOPIC)
df_target_exact = df_rule.loc[doc_info.loc[mask_exact, 'row_id']].copy()
df_target_exact['topic_assigned'] = TARGET_TOPIC


In [36]:
df_target_exact

,id,message_id,date,message,id_channel,message_vector,msg_clean,topic_assigned
683,7d7daae1-923b-49ad-963b-d678330c3e55,274806,2024-12-26 11:03:16,Центральный банк Турции впервые почти за два г...,3,"'1':46 '2024':18 '47':13,45 '5':14 '50':23 'ба...",Центральный банк Турции впервые почти за два г...,60
1164,75a26d0b-8833-436b-b6a9-13ccaa8f8bae,274296,2024-12-23 08:04:23,"Москвичи считают, что оливье надо готовить с в...",3,'18':33 '24':29 '46':21 'активн':16 'варен':8 ...,"Москвичи считают, что оливье надо готовить с в...",60
2602,859b5828-f92d-4b49-8248-5b3be13b07d3,272710,2024-12-12 12:18:51,В России за последние годы стали меньше курить...,3,"'15':18 '18':28 '2':26 '24':25 '6':29,38 '7':3...",В России за последние годы стали меньше курить...,60
2641,de98f89e-16b5-4aba-b5d7-6811f6c6fa43,272666,2024-12-12 07:43:21,Более четверти россиян (27%) считают хоккей на...,3,'27':4 '9':23 'втор':13 'вци':12 'гимнастик':3...,Более четверти россиян (27%) считают хоккей на...,60
2791,e8ba0d41-3d8b-41d3-b3ac-7d247e258f31,272512,2024-12-11 01:31:03,Среди всех российских регионов сильнее всего н...,3,'18':19 '19':15 '3':29 '4':30 'автономн':27 'к...,Среди всех российских регионов сильнее всего н...,60
...,...,...,...,...,...,...,...,...
106969,68a07fe0-0c4a-4105-bb8c-1a969870865e,84044,2025-07-25 10:30:36,❗Совет директоров Банк России на июльском засе...,6,'18':12 'банк':3 'втор':15 'директор':2 'замед...,❗Совет директоров Банк России на июльском засе...,60
109669,fea72b02-adb8-47b0-b1d1-f205aeb1f246,308108,2025-08-03 02:41:54,Только 29% британцев сейчас бы одобрили Brexit...,3,'2':13 '2016':25 '22':17 '24':19 '29':2 '52':3...,Только 29% британцев сейчас бы одобрили Brexit...,60
110403,d622b8e3-cbf5-4b23-8fc3-501ff337d139,8915,2025-08-05 07:20:06,Зарплаты мастеров маникюра в России взлетели д...,5,'130':8 '50':18 'bloomberg':40 'взлетел':6 'вы...,Зарплаты мастеров маникюра в России взлетели д...,60
119089,c31db082-b7e0-464c-8bed-c94ffa233f0b,85375,2025-08-27 13:45:21,Совет директоров «Полюса» рекомендовал акционе...,6,"'2025':13 '3':28,29 '70':17 '85':18 'акц':21,2...",Совет директоров «Полюса» рекомендовал акционе...,60


In [37]:
df_target_exact.to_excel("60_topic.xlsx", index=False)


In [39]:
import pandas as pd
from pathlib import Path

# 1) Документная инфа (всё уже внутри doc_info)
doc_info = topic_model.get_document_info(news).copy()

# 2) Привязываем строки к исходному df_rule и подтягиваем текст
doc_info['row_id'] = df_rule.index.to_numpy()
doc_info['message'] = df_rule.loc[doc_info['row_id'], 'message'].astype(str).values

# 3) Переименования столбцов под итоговый паспорт
rename_map = {
    'Topic': 'topic_id',
    'Name': 'topic_name'
}
doc_info.rename(columns=rename_map, inplace=True)

# 4) Аккуратно выбираем нужные колонки (только те, что реально есть в doc_info)
wanted = [
    'row_id',
    'topic_id',
    'topic_name',
    'Representation',
    'KeyBERT',
    'MMR',
    'POS',
    'DeepSeek',
    'message'
]
cols = [c for c in wanted if c in doc_info.columns]
full = doc_info[cols].copy()

# (опционально) добавим любые поля из df_rule, если есть
for extra in ['date', 'channel_id', 'source', 'url']:
    if extra in df_rule.columns:
        full[extra] = df_rule.loc[full['row_id'], extra].values

# 5) Сохраняем
out_dir = Path("outputs"); out_dir.mkdir(parents=True, exist_ok=True)
csv_path  = out_dir / "news_with_topics.csv"
xlsx_path = out_dir / "news_with_topics.xlsx"

full.to_csv(csv_path, index=False)
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    full.to_excel(writer, index=False, sheet_name="news_topics")

print(f"Готово! Строк: {len(full):,}\nCSV:  {csv_path.resolve()}\nXLSX: {xlsx_path.resolve()}")


Готово! Строк: 68,706
CSV:  /home/dasha/work/narrative/topics/outputs/news_with_topics.csv
XLSX: /home/dasha/work/narrative/topics/outputs/news_with_topics.xlsx


In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv("topics_scored_ranking.csv")

In [3]:
df

,Topic,Name,Count,trigger,causal,emo,state,media,simple,snowball,trigger_n,causal_n,emo_n,state_n,media_n,simple_n,snowball_n,Score,eligible
0,119,119_goldman_goldman sachs_sachs_аналитики goldman,39,1,1,1,1,1,0,1,3,1,3,3,3,0,3,21.0,True
1,4,4_коммерсантъ_пишет коммерсантъ_объявил_объектов,1338,1,1,1,1,1,0,1,3,1,3,3,3,0,3,21.0,True
2,103,103_genius_weather_трамп_запретить,51,1,1,0,1,1,1,1,3,1,0,2,3,2,3,17.5,True
3,2,2_цб_цен_цены_центр,1652,1,0,1,1,0,1,1,3,0,1,3,0,2,3,17.0,True
4,150,150_газа_рф_нa_головы,31,1,1,1,1,1,1,1,3,1,1,2,3,1,2,16.5,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,88,88_минобороны_освободили_освобождении_военные ...,57,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,True
169,78,78_bitcoin_000_биткоин_унцию,67,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,True
170,69,69_пво_украинских_курской_минобороны,80,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,True
171,44,44_курском направлении_всу курском_направлении...,124,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,True


In [4]:
df[['Name', "Count", "Score"]]

,Name,Count,Score
0,119_goldman_goldman sachs_sachs_аналитики goldman,39,21.0
1,4_коммерсантъ_пишет коммерсантъ_объявил_объектов,1338,21.0
2,103_genius_weather_трамп_запретить,51,17.5
3,2_цб_цен_цены_центр,1652,17.0
4,150_газа_рф_нa_головы,31,16.5
...,...,...,...
168,88_минобороны_освободили_освобождении_военные ...,57,0.0
169,78_bitcoin_000_биткоин_унцию,67,0.0
170,69_пво_украинских_курской_минобороны,80,0.0
171,44_курском направлении_всу курском_направлении...,124,0.0


In [4]:
df.to_excel('topics_scored_ranking.xlsx', index=False)